# TF-IDF baseline on Colab

Runs the heavy part of stage 3 (`scripts/tune_tfidf.py`) on a Colab **CPU** runtime, which has
about 12 GB of memory. No GPU is needed, so no GPU quota is used.

- The code comes from GitHub at one exact commit.
- The normalized texts come from Google Drive and are checked against the hashes recorded
  when they were built locally. Only train and validation are copied; test stays at home.
- Every result is written to Google Drive as soon as it is ready. If the runtime stops, run
  the cells again: finished runs are skipped. Any failing step stops the notebook.

Before starting: push the code, copy `train_supervised.parquet`, `validation.parquet` and
`cache_info.json` from `data/cache/` to `MyDrive/digikala/cache/`, and put the commit below.

In [ ]:
import os
import re
import subprocess

COMMIT = "PUT-THE-COMMIT-HERE"  # on your machine: git log -1 --format=%H (after git push)
DRIVE_FOLDER = "/content/drive/MyDrive/digikala"  # holds cache/ and receives outputs/

REPO_URL = "https://github.com/alidarviish/digikala-review-analysis.git"
OUTPUTS = f"{DRIVE_FOLDER}/outputs"
LOGS = f"{OUTPUTS}/reports/logs"
assert re.fullmatch(r"[0-9a-f]{40}", COMMIT), "put the full 40-character commit above"


def run(command):
    """Run a shell command, show its output as it comes, and stop the notebook if it fails."""
    process = subprocess.Popen(
        ["bash", "-c", "set -o pipefail; " + command],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(f"Failed: {command}")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
run(f"rm -rf /content/repo && git clone -q {REPO_URL} /content/repo")
run(f"git -C /content/repo checkout -q {COMMIT} && git -C /content/repo log -1 --oneline")

Install the exact library versions of the project. The scripts run as separate processes,
so they use these versions even if Colab has others preinstalled.

In [ ]:
run("pip install -q -r /content/repo/requirements.txt && pip install -q --no-deps -e /content/repo")
run("python -c 'import sklearn, pandas; print(sklearn.__version__, pandas.__version__)'")
run("free -h")

Copy the cache to the runtime's own disk (reading from Drive is slow), and tell the scripts
where the cache is and where to write the outputs.

In [ ]:
run("mkdir -p /content/cache")
for name in ["train_supervised.parquet", "validation.parquet", "cache_info.json"]:
    run(f"cp {DRIVE_FOLDER}/cache/{name} /content/cache/")
os.environ["DIGIKALA_CACHE_DIR"] = "/content/cache"
os.environ["DIGIKALA_OUTPUT_DIR"] = OUTPUTS
os.makedirs(LOGS, exist_ok=True)
run("ls -la /content/cache")

## Stage 1: feature setups

Every setup with C = 1, with and without class weights, for both heads. Several hours;
the log is also written to `outputs/reports/logs/` on Drive. Keep this browser tab open.

In [ ]:
TUNE = "cd /content/repo && python scripts/tune_tfidf.py"
run(f"{TUNE} --stage features 2>&1 | tee -a {LOGS}/stage1.log")

## Stage 2: regularization

For each head, its best setup from stage 1 with other values of C. It refuses to start
until every stage 1 run is finished.

In [ ]:
TUNE = "cd /content/repo && python scripts/tune_tfidf.py"
run(f"{TUNE} --stage regularization 2>&1 | tee -a {LOGS}/stage2.log")

## Close the session

List every output file with its hash and make sure everything has reached Drive. Then
download `MyDrive/digikala/outputs` and, on your machine, run
`uv run python scripts/import_colab_outputs.py <downloaded folder>`.

In [ ]:
run("cd /content/repo && python scripts/write_outputs_manifest.py")
drive.flush_and_unmount()